# Silver — monthly performance

`bronze.trust_prices_yf` + `bronze.index_prices` + the two archived trusts from
`bronze.trust_prices_csv` → **`silver.monthly_performance`**, plus the audit table
**`silver.price_repair_log`**.

This is where the raw bars become the thing the project measures: a monthly return series
for every trust and for the index, built the same way on both sides.

One principle runs through it: **where a price cannot be trusted, it is deleted rather than
guessed at** — unless the source's own metadata says exactly what is wrong with it, which is
what the split repair does. That applies to a single month, and to five trusts whose whole
early history is recorded in two different units.

Spec: `specs/02_silver/silver.md`, amended by `specs/02_silver/split-repair.md`. Run this
**before** `silver_etl_ticker`, which reads the coverage facts back out of the table this
notebook writes.

Expected: **15,604 rows** across **96 tickers**, and a repair log of **346 rows**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.silver.monthly_performance (
  ticker       STRING  COMMENT 'Business key. No .L suffix',
  month_key    INT     COMMENT 'YYYYMM. Readable and sorts naturally, so not a hash',
  month_start  DATE,
  close        DOUBLE  COMMENT 'Repaired where Bronze recorded an impossible price',
  dividend     DOUBLE,
  price_return DOUBLE  COMMENT 'Null across a gap rather than computed over two months',
  total_return DOUBLE  COMMENT 'Null for the archive trusts: the CSV has no dividends',
  currency     STRING  COMMENT 'Carried, never converted. A return is unit-free',
  price_source STRING,
  is_repaired  BOOLEAN,
  return_basis STRING  COMMENT 'Gold must never compare a price return to a total return'
)
COMMENT 'Monthly price and total return for every trust and the index, 2011-08 onward';

## The repair chain

Four steps, each a CTE, applied identically to trusts and to the index.

1. **Detect.** Two independent tests, either one is enough: the close sits more than **2x**
   from the median of its 13-month neighbourhood, **or** more than **25%** outside its own
   bar's high–low, which is arithmetically impossible.
2. **Repair.** Three candidates, tried in order, and **each one has to land back inside the
   detector's band to win**:
   - the month's own `(High + Low) / 2` — a price the instrument genuinely traded at, used
     only when the bar is coherent. If the high is more than twice the low, those two
     numbers are recorded in different units and their mid-point is not a price at all.
   - `Close / F`, where `F` is the product of the splits **Yahoo itself reports** dated
     after this month. Yahoo left four quarter-ends on the pre-split scale for 22 trusts,
     and its own `Stock_Splits` column says by exactly how much.
   - the nearest power of ten, for the archive, which has no bar at all.
3. **Re-test.** Run the chosen value back through the detector.
4. **Delete.** If no candidate passed, the month goes.

Candidates rather than a single fall-through `CASE`, because a `CASE` cannot retry: a
coherent bar whose high, low *and* close are all pre-split would match the mid-point branch,
produce a pre-split value and die, without the split candidate ever being tried.

The neighbourhood median spans each ticker's **full** history and the study window is
filtered afterwards, so the earliest months are not judged against a truncated half-window.

In [ ]:
CREATE OR REPLACE TEMP VIEW silver_stage_prices AS
WITH partial AS (
  -- The month the pull ran is incomplete, so it is excluded. Derived, never hardcoded.
  SELECT TRUNC(MAX(CAST(pulled_at AS TIMESTAMP)), 'MM') AS partial_month
  FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
),
src AS (
  SELECT REPLACE(symbol, '.L', '')                      AS ticker,
         'Trust'                                        AS entity_type,
         'yahoo'                                        AS price_source,
         TRUNC(TO_DATE(SUBSTRING(`Date`, 1, 10)), 'MM') AS month_start,
         CAST(`High` AS DOUBLE)                         AS high,
         CAST(`Low` AS DOUBLE)                          AS low,
         CAST(`Close` AS DOUBLE)                        AS close_raw,
         CAST(`Dividends` AS DOUBLE)                    AS dividend,
         -- Yahoo's own split column. NULL in the months with no split.
         CASE WHEN CAST(`Stock_Splits` AS DOUBLE) > 1
              THEN CAST(`Stock_Splits` AS DOUBLE) END   AS split_ratio
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf

  UNION ALL

  SELECT symbol, 'Index', 'yahoo',
         TRUNC(TO_DATE(SUBSTRING(`Date`, 1, 10)), 'MM'),
         CAST(`High` AS DOUBLE), CAST(`Low` AS DOUBLE),
         CAST(`Close` AS DOUBLE), CAST(`Dividends` AS DOUBLE),
         CASE WHEN CAST(`Stock_Splits` AS DOUBLE) > 1
              THEN CAST(`Stock_Splits` AS DOUBLE) END
  FROM `index-vs-trust-pipeline`.bronze.index_prices

  UNION ALL

  -- The two delisted trusts Yahoo erased. Price only: no high, low, dividend or split.
  -- CSV dates render month-end as next-month-first, so day <= 3 belongs to the month before.
  SELECT ticker, 'Trust', 'archive',
         CASE WHEN CAST(SUBSTRING(date, 9, 2) AS INT) <= 3
              THEN ADD_MONTHS(TRUNC(TO_DATE(SUBSTRING(date, 1, 10)), 'MM'), -1)
              ELSE TRUNC(TO_DATE(SUBSTRING(date, 1, 10)), 'MM') END,
         NULL, NULL, CAST(price_gbx_or_gbp AS DOUBLE), NULL, NULL
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
  WHERE ticker IN ('BCPT', 'CSH')
),
scored AS (
  SELECT src.*,
         PERCENTILE_APPROX(close_raw, 0.5) OVER (
           PARTITION BY ticker, price_source
           ORDER BY month_start
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median,
         -- Every split dated after this month, multiplied together. A month priced before
         -- two later splits is out by both. EXP(SUM(LN())) because Spark has no product
         -- aggregate, and SUM skips the NULLs so a ticker that never split comes out 1.
         COALESCE(EXP(SUM(LN(split_ratio)) OVER (
           PARTITION BY ticker, price_source
           ORDER BY month_start
           ROWS BETWEEN 1 FOLLOWING AND UNBOUNDED FOLLOWING
         )), 1) AS split_factor
  FROM src
),
flagged AS (
  SELECT scored.*,
         CASE WHEN close_raw <= 0
                -- Against its neighbours.
                OR close_raw / local_median > 2
                OR close_raw / local_median < 0.5
                -- Against itself: a close outside its own high-low cannot have traded.
                -- The 25% margin skips the small noise in Yahoo's monthly aggregation.
                OR (high > 0 AND (close_raw > high * 1.25 OR close_raw < low * 0.75))
              THEN TRUE ELSE FALSE END AS is_flagged
  FROM scored
  WHERE local_median > 0 AND close_raw IS NOT NULL
),
candidates AS (
  SELECT flagged.*,
         -- The bar's own mid-point: a price that genuinely traded that month. Only from a
         -- coherent bar: when the high is more than twice the low the two are recorded in
         -- different units and their mid-point is not a price.
         CASE WHEN high > 0 AND low > 0 AND high / low <= 2
              THEN (high + low) / 2 END                 AS mid_candidate,
         -- Yahoo left this month on the pre-split scale. Its own split column says by how
         -- much, so this is read from the source's metadata rather than estimated.
         CASE WHEN split_factor > 1
              THEN close_raw / split_factor END         AS split_candidate,
         -- The archive has no bar, so fall back to the unit error CSH actually has.
         CASE WHEN high IS NULL AND close_raw > 0
              THEN close_raw / POWER(10, ROUND(LOG10(close_raw / local_median)))
         END                                            AS pot_candidate
  FROM flagged
),
chosen AS (
  -- First candidate that lands back inside the detector's band wins. NULL comparisons are
  -- false, so a missing candidate falls through instead of erroring.
  SELECT candidates.*,
         CASE WHEN NOT is_flagged THEN close_raw
              WHEN mid_candidate   / local_median BETWEEN 0.5 AND 2 THEN mid_candidate
              WHEN split_candidate / local_median BETWEEN 0.5 AND 2 THEN split_candidate
              WHEN pot_candidate IS NOT NULL                        THEN pot_candidate
              -- Nothing trustworthy left; NULL sends the month to deletion.
         END AS close_fixed,
         CASE WHEN NOT is_flagged THEN NULL
              WHEN mid_candidate   / local_median BETWEEN 0.5 AND 2 THEN 'bar-midpoint'
              WHEN split_candidate / local_median BETWEEN 0.5 AND 2 THEN 'split'
              WHEN pot_candidate IS NOT NULL                        THEN 'power-of-ten'
         END AS repair_method
  FROM candidates
)
SELECT chosen.*,
       -- Re-test. Still wrong means the whole bar is corrupt, so the month goes.
       CASE WHEN is_flagged
                 AND (close_fixed IS NULL
                      OR NOT (close_fixed / local_median BETWEEN 0.5 AND 2))
            THEN TRUE ELSE FALSE END AS is_deleted
FROM chosen
WHERE month_start >= DATE'2011-08-01'
  AND month_start <  (SELECT partial_month FROM partial);

In [0]:
-- Trusts whose price series cannot be trusted as a whole. Two causes, one principle:
-- where a price cannot be trusted it is dropped, never guessed at.
CREATE OR REPLACE TEMP VIEW silver_stage_excluded AS
WITH splits AS (
  SELECT REPLACE(symbol, '.L', '') AS ticker,
         MAX(CAST(`Stock_Splits` AS DOUBLE)) AS split_ratio
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  WHERE CAST(`Stock_Splits` AS DOUBLE) > 1
  GROUP BY 1
),
levels AS (
  SELECT p.ticker, s.split_ratio,
         PERCENTILE_APPROX(p.close_fixed, 0.5) OVER (
           PARTITION BY p.ticker ORDER BY p.month_start
           ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS lvl_before,
         PERCENTILE_APPROX(p.close_fixed, 0.5) OVER (
           PARTITION BY p.ticker ORDER BY p.month_start
           ROWS BETWEEN CURRENT ROW AND 5 FOLLOWING) AS lvl_after
  FROM silver_stage_prices p
  JOIN splits s ON s.ticker = p.ticker
  WHERE NOT p.is_deleted
),
moves AS (
  SELECT ticker, month_start, close_fixed,
         LAG(close_fixed)  OVER (PARTITION BY ticker ORDER BY month_start) AS prev_close,
         LAG(month_start)  OVER (PARTITION BY ticker ORDER BY month_start) AS prev_month
  FROM silver_stage_prices
  -- The archive is exempt: its two trusts are the only recoverable evidence of
  -- survivorship, so their weakness is stated rather than used to delete them.
  WHERE NOT is_deleted AND entity_type = 'Trust' AND price_source = 'yahoo'
)
-- 1. The level steps by the trust's own recorded split ratio: the signature of a
--    back-adjustment Yahoo applied to only part of the history. Requiring the trust's own
--    ratio is what separates it from a genuine collapse, which has no split behind it.
SELECT DISTINCT ticker
FROM levels
WHERE lvl_before > 0 AND lvl_after > 0
  AND ABS(lvl_before / lvl_after - split_ratio) / split_ratio <= 0.35

UNION

-- 2. A dividend larger than half the share price in one month. No trust pays that, so the
--    close and the dividend are recorded in different units and we cannot tell which is
--    wrong. CLDN reaches 642% of its price this way.
SELECT DISTINCT ticker
FROM silver_stage_prices
WHERE NOT is_deleted
  AND entity_type = 'Trust'
  AND close_fixed > 0
  AND dividend / close_fixed > 0.5

UNION

-- 3. A month-on-month move still beyond +100% or -50% after every repair. No trust moves
--    that far in a month, so the series is still mixing units somewhere we cannot locate.
SELECT DISTINCT ticker
FROM moves
WHERE prev_month = ADD_MONTHS(month_start, -1)
  AND prev_close > 0
  AND (close_fixed / prev_close > 2 OR close_fixed / prev_close < 0.5);

In [ ]:
-- Overwritten each run: it describes this run's repairs, not history.
CREATE OR REPLACE TABLE `index-vs-trust-pipeline`.silver.price_repair_log
COMMENT 'Every price Silver changed or removed, with the evidence for the decision'
AS
SELECT ticker,
       CAST(DATE_FORMAT(month_start, 'yyyyMM') AS INT)      AS month_key,
       month_start,
       price_source,
       close_raw                                            AS close_before,
       CASE WHEN is_deleted THEN NULL ELSE close_fixed END  AS close_after,
       high,
       low,
       ROUND(local_median, 4)                               AS neighbour_median,
       ROUND(close_raw / local_median, 3)                   AS ratio_to_neighbours,
       -- The split Yahoo reports after this month, so the repair can be re-derived.
       CASE WHEN repair_method = 'split' THEN ROUND(split_factor, 4) END AS split_factor,
       CASE WHEN high > 0 AND (close_raw > high OR close_raw < low)
            THEN TRUE ELSE FALSE END                        AS close_outside_own_bar,
       CASE WHEN is_deleted                 THEN 'deleted'
            WHEN repair_method = 'split'    THEN 'split-repaired'
            ELSE 'repaired' END                             AS outcome,
       CASE WHEN close_raw <= 0                 THEN 'close recorded as zero'
            WHEN is_deleted                     THEN 'repair failed re-test: whole bar corrupt'
            WHEN repair_method = 'split'
                 THEN 'left on the pre-split scale, divided by the split Yahoo reports'
            WHEN repair_method = 'power-of-ten' THEN 'unit error in the archive, rescaled by a power of ten'
            WHEN close_raw > high OR close_raw < low
                 THEN 'close outside its own high-low, replaced with the bar mid-point'
            ELSE 'far from neighbouring months, replaced with the bar mid-point'
       END                                                  AS reason
FROM silver_stage_prices
WHERE ticker NOT IN (SELECT ticker FROM silver_stage_excluded)
  AND is_flagged;

In [0]:
CREATE OR REPLACE TEMP VIEW silver_stage_monthly AS
WITH kept AS (
  SELECT ticker, price_source, month_start,
         close_fixed AS close, dividend, is_flagged AS is_repaired
  FROM silver_stage_prices
  WHERE NOT is_deleted
    AND ticker NOT IN (SELECT ticker FROM silver_stage_excluded)
),
seq AS (
  SELECT kept.*,
         LAG(close)       OVER (PARTITION BY ticker ORDER BY month_start) AS prev_close,
         LAG(month_start) OVER (PARTITION BY ticker ORDER BY month_start) AS prev_month
  FROM kept
),
ccy AS (
  -- One row per ticker, so the join cannot fan out.
  SELECT source_ticker, MAX(currency) AS currency
  FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
  GROUP BY source_ticker
)
SELECT seq.ticker,
       CAST(DATE_FORMAT(seq.month_start, 'yyyyMM') AS INT) AS month_key,
       seq.month_start,
       seq.close,
       seq.dividend,
       -- Null, not zero, when the previous month is missing. Compounding across a hole
       -- would turn a two-month move into a one-month return and inflate volatility.
       CASE WHEN seq.prev_month = ADD_MONTHS(seq.month_start, -1) AND seq.prev_close > 0
            THEN seq.close / seq.prev_close - 1 END         AS price_return,
       CASE WHEN seq.prev_month = ADD_MONTHS(seq.month_start, -1) AND seq.prev_close > 0
                 AND seq.dividend IS NOT NULL
            THEN (seq.close + seq.dividend) / seq.prev_close - 1 END AS total_return,
       ccy.currency,
       seq.price_source,
       seq.is_repaired,
       -- The archive has no dividends, so those two trusts can only be price return.
       CASE WHEN seq.price_source = 'archive' THEN 'price' ELSE 'total' END AS return_basis
FROM seq
LEFT JOIN ccy ON ccy.source_ticker = seq.ticker;

In [0]:
-- MERGE on the business key: re-running adds new months and corrects restated ones
-- without ever duplicating a row. A month's verdict can firm up as the six-month
-- forward window fills, which is why matched rows are updated rather than skipped.
MERGE INTO `index-vs-trust-pipeline`.silver.monthly_performance AS t
USING silver_stage_monthly AS s
   ON t.ticker = s.ticker AND t.month_key = s.month_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## Verification

Every expected answer below is stated in `specs/02_silver/silver.md` and was measured
against the warehouse before this notebook was written.

In [0]:
SELECT COUNT(*)                                                            AS rows_total,
       COUNT(DISTINCT ticker)                                              AS tickers,
       MIN(month_key)                                                      AS first_month,
       MAX(month_key)                                                      AS last_month,
       SUM(CASE WHEN ticker IN ('SPY','IVV','VOO') THEN 1 ELSE 0 END)      AS index_rows,
       SUM(CASE WHEN price_source = 'archive'      THEN 1 ELSE 0 END)      AS archive_rows,
       SUM(CASE WHEN ticker = 'MNTN'               THEN 1 ELSE 0 END)      AS mntn_rows
FROM `index-vs-trust-pipeline`.silver.monthly_performance;

Expect **15,604 / 96 / 201108 / 202608 / 543 / 270 / 0**.

Yahoo trust rows are the remainder, **14,791**. `MNTN` is absent because its only bar sits in
the excluded partial month, and seven more trusts are absent for the reason below.

The total was **15,516** before the split repair. The extra **88** rows are the four
quarter-ends Yahoo left on the pre-split scale for 22 trusts, which used to be deleted.

In [0]:
SELECT outcome,
       COUNT(*)               AS rows,
       COUNT(DISTINCT ticker) AS tickers
FROM `index-vs-trust-pipeline`.silver.price_repair_log
GROUP BY outcome
ORDER BY outcome;

Expect three outcomes — **346 rows in total**, unchanged, because detection did not change;
only what happens to a flagged row did:

| outcome | rows | tickers |
|---|---|---|
| `deleted` | **6** | **2** |
| `repaired` | **252** | **18** |
| `split-repaired` | **88** | **22** |

Before the split repair this read *deleted 94 / 24* and *repaired 252 / 18*. The **6** that
still go are the control group: **1 `BSIF`** row whose ratio is 90 with no split reported,
and **5 `JEGI`** rows that span 2011-12 to 2015-08 rather than the four quarter-ends and
have no split behind them either. **A rule that rescued those too would be curve-fitting.**

In [0]:
-- The two checks that prove the repair worked, by coming back empty.
WITH rechecked AS (
  SELECT close,
         PERCENTILE_APPROX(close, 0.5) OVER (
           PARTITION BY ticker ORDER BY month_key
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
)
SELECT SUM(CASE WHEN close <= 0 THEN 1 ELSE 0 END)                            AS non_positive_close,
       SUM(CASE WHEN close / local_median > 2
                  OR close / local_median < 0.5 THEN 1 ELSE 0 END)            AS still_out_of_scale
FROM rechecked
WHERE local_median > 0;

Expect **0 / 0**.

`still_out_of_scale` is the check that earned its keep: on the first dry run it returned
**1**, which exposed `JEMA` 2022-12 (`low 79.75, high 90.00, close 42.87`) slipping past a
detector that only looked at neighbouring months. That is why rule **S2b** exists.

In [ ]:
-- Named rows whose repaired value is known in advance.
SELECT ticker, month_key, ROUND(close, 3) AS close, is_repaired, return_basis, currency
FROM `index-vs-trust-pipeline`.silver.monthly_performance
WHERE (ticker = 'PCFT' AND month_key = 201911)
   OR (ticker = 'FCIT' AND month_key = 202505)
   OR (ticker = 'JEMA' AND month_key = 202212)
   OR (ticker = 'BSIF' AND month_key = 202608)
   OR (ticker = 'ATT'  AND month_key = 201112)
   OR (ticker = 'SMT'  AND month_key = 201112)
ORDER BY ticker;

Expect **5 rows**:

| ticker | month | close | was | why |
|---|---|---|---|---|
| `ATT` | 201112 | **29.700** | 297.00 | left on the pre-split scale; ÷10, the split Yahoo reports |
| `BSIF` | 202608 | **1.023** | 1.023 | untouched — its bar has `H=0`, nothing to repair from |
| `FCIT` | 202505 | **267.875** | 1133.87 | above its own high of 278.00 |
| `PCFT` | 201911 | **141.250** | 0.0 | the zero price, from `H=145.0 L=137.5` |
| `SMT` | 201112 | **118.000** | 590.00 | left on the pre-split scale; ÷5 |

`ATT` and `SMT` are the split repair: both were **deleted** before it, and the untouched
`Low` on those bars is 29.60 and 112.01 — the repaired close lands on the number that was
already sitting in the row.

`BSIF` is the guard working: a bar with no high has nothing to repair from, so it is left
alone rather than "repaired" to a close of zero.

**`JEMA` is deliberately not in this list.** It is excluded as a whole trust — an impossible
month survived every repair — so it cannot appear in `monthly_performance` at all. An earlier
version of this cell expected it here, which was wrong.

In [ ]:
-- The split repair, checked against a second provider. The CSV archive carries the same
-- half-applied split defect in *different* months, so where it is sound it is an
-- independent opinion on the scale rather than a copy of Yahoo's.
WITH arch AS (
  SELECT ticker,
         CASE WHEN CAST(SUBSTRING(date, 9, 2) AS INT) <= 3
              THEN ADD_MONTHS(TRUNC(TO_DATE(SUBSTRING(date, 1, 10)), 'MM'), -1)
              ELSE TRUNC(TO_DATE(SUBSTRING(date, 1, 10)), 'MM') END AS month_start,
         AVG(CAST(price_gbx_or_gbp AS DOUBLE))                      AS archive_price
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
  GROUP BY 1, 2
)
SELECT COUNT(*)                                                                  AS split_repairs,
       SUM(CASE WHEN ABS(l.close_after  / a.archive_price - 1) <= 0.10
                THEN 1 ELSE 0 END)                                               AS agree_after_repair,
       SUM(CASE WHEN ABS(l.close_before / a.archive_price - 1) <= 0.10
                THEN 1 ELSE 0 END)                                               AS agreed_before_repair,
       ROUND(PERCENTILE_APPROX(ABS(l.close_after / a.archive_price - 1), 0.5), 4) AS median_error
FROM `index-vs-trust-pipeline`.silver.price_repair_log l
JOIN arch a ON a.ticker = l.ticker AND a.month_start = l.month_start
WHERE l.outcome = 'split-repaired';

Expect **88 / 87 / 0 / 0.0242** — every split-repaired row has an archive price to check
against, **87 of the 88** agree with it within 10% after the repair, and **none** agreed
before. Median error afterwards is **2.4%**, which is two providers' month-end conventions
differing, not a scale problem.

This is the strongest evidence in the project. The repair is not *"my rule said so"*: a
second, independent source — corrupted in different months, so not a copy — agrees with 87
of the 88 repaired prices and with none of the originals.

The single exception is `SMT` 2011-12, out by 10.4%. It is left visible rather than
disappeared by widening the threshold.

In [0]:
-- CSH mixed pounds and pence, which would have wrecked half the delisted cohort.
-- Both archive trusts are price-return only, because the CSV carries no dividends.
SELECT ticker,
       COUNT(*)                                            AS months,
       ROUND(MIN(close), 3)                                AS min_close,
       ROUND(MAX(close), 3)                                AS max_close,
       SUM(CASE WHEN total_return IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_total_return,
       MAX(return_basis)                                   AS basis
FROM `index-vs-trust-pipeline`.silver.monthly_performance
WHERE price_source = 'archive'
GROUP BY ticker
ORDER BY ticker;

Expect `BCPT` **158 months** and `CSH` **112 months**, both with **0** rows carrying a total
return and a basis of **price**. `CSH`'s max close must be **2.945** — before the repair it
was 112.8, because ten of its months were quoted in pence and the rest in pounds.

In [0]:
-- The five trusts dropped whole, and why. Yahoo applied each split's back-adjustment to
-- only part of the history, so the early months are in the pre-split unit and the later
-- months in the post-split one.
SELECT e.ticker,
       s.split_ratio,
       COUNT(*)            AS yahoo_months_offered,
       MIN(p.month_start)  AS from_month,
       MAX(p.month_start)  AS to_month
FROM silver_stage_excluded e
JOIN silver_stage_prices p ON p.ticker = e.ticker
JOIN (SELECT REPLACE(symbol,'.L','') AS ticker,
             MAX(CAST(`Stock_Splits` AS DOUBLE)) AS split_ratio
      FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
      WHERE CAST(`Stock_Splits` AS DOUBLE) > 1 GROUP BY 1) s ON s.ticker = e.ticker
GROUP BY e.ticker, s.split_ratio
ORDER BY e.ticker;

Expect **7 rows**. Three different reasons, one principle — *where a price cannot be trusted
it is dropped, never guessed at*:

| trusts | why |
|---|---|
| `MRC`, `MYI`, `NAS`, `PCT`, `WWH` | Yahoo applied their split back-adjustment to only part of the history |
| `CLDN` | dividends reach **642%** of the share price, so price and dividend are in different units |
| `JEMA` | an impossible month survived every repair |

Left in, `WWH` reports a 15-year return of **−50.9%** when the truthful figure is about
**+356%**, and `CLDN` compounds to **91 million per cent**, topping every leaderboard.

These are detected, not hardcoded. The split test requires the level to step by the trust's
**own recorded split ratio**, which is what separates a half-applied adjustment from a real
crash — `CHRY` fell 75% across 2022 in a smooth ten-month slide with no split behind it, and
is correctly left alone.

`EOT` is also dropped by the dividend test, but it was already a two-month stub.